# பாடம் 18 (தொடர்ச்சி): ஒரு *மனிதர்* நடவடிக்கையை அங்கீகரித்ததைக் காட்டும் ரசீத்கள்

படிப்பு **நடவடிக்கை செய்தவர்** என்ன செய்தார் மற்றும் **கதவு** என்ன முடிவு செய்தது என்பதை நிரூபிக்கிறது. இதில் காணப்படும் நோட்புக் காணாமல் போன பாதியை சேர்க்கிறது: **பெயரிட்ட மனிதர்** சரியான **நடவடிக்கையை** அங்கீகரித்ததை நிரூபிக்கும் — முழு அதிகாரப்பூர்வ நடவடிக்கையில் தனித்தனி மனித கையொப்பம், ஆன்லைனுக்கு உட்படாமல் சரிபார்க்கப்பட்டது.

இங்கு உள்ள இரு கலைநுட்பங்கள் பாடத்திற்கான ரசீத்களின் **ஏதையுமில்லாத அட்டுப்படி வடிவத்தை** பயன்படுத்துகின்றன: ஒரு நிலையான தகவலுடன் `type` புலம், canonical JCS பைட்ஸ் மீது நேரடியாக Ed25519 கையொப்பம் உடன், அமைப்பான `signature` பொருள் இணைத்து (கையொப்பப்பட்ட பைட்ஸில் இருந்து விலக்கப்பட்டுள்ளது). அங்கீகார ரசீதம் புதிய `type` (`human.approval.v1`) ஆகும், நடவடிக்கை வகை உடன் இணைத்து, அதனால் ஒரே `verify_chain` இரண்டு கலைநுட்ப வகைகளையும் ஒரே கோட் பாதை மூலம் பராமரிக்கிறது, நீங்கள் முதன்மையான நோட்புக்கில் உருவாக்கியது. இந்த மனித அங்கீகார ரசீதம் இங்கே வரையறுக்கப்பட்ட கல்வி படைப்பாகும், draft-farley-acta-signed-receipts மூலம் வரையறுக்கப்படாத ரசீத வகை ஆகும்.

முதன்மை நோட்புக்கில் உள்ள டெமோ சரிபார்ப்பாளருக்கு மேலே செய்யப்பட்ட மாற்றம் ஒன்று: இங்கு சரிபார்ப்பாளர் `signature.key_id` ஐ, ரசீதத்தில் உள்ள பொதுக்கீவை நம்புவதற்கு பதிலாக, **பிணைத்துக் கொண்ட விசை பதிவேட்டின்** முறைப்படி தீர்மானிக்கின்றார். இது பாடத்தின் சொந்த சரிபார்ப்பு பட்டியலில் பரிந்துரைக்கப்படும் உற்பத்தி நிலை நடைமுறை ("சரிபார்ப்பு பொது விசையை வெளியிடுக"), மேலும் இது போலியானதை மறுத்து, உங்கள் விசையை கொண்டு செல்லும் வழியை தடுக்கும்.

இந்த நோட்புக் கற்பிக்கும் விதி: **கையொப்பமிட்ட அங்கீகாரம் தனக்கென அதிகாரம் அல்ல.** அங்கீகார ரசீதமும் நடவடிக்கை ரசீதமும் ஒரே canonical நடவடிக்கையை தொடர்ந்துபிடிக்க வேண்டும் நிகழ்ச்சிக்கான நேரத்தில், கொள்கை பதிப்பு, விசை, மற்றும் காலாவதி நிலை இன்னும் செல்லுபடியாக இருக்க வேண்டும், மற்றும் ஒரு அங்கீகாரம் ஏற்கப்படவில்லை என்பது உறுதி செய்ய வேண்டும். ஒவ்வொரு தங்கை தோல்வியும் வேறுவித காரணத்துடன் மறுக்கப்படுகிறது, எனவே *அதிகாரம் காலாவதியானது* மற்றும் *நடவடிக்கை மாற்றப்பட்டது* என்பதை வேறுபடுத்த முடியும்.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## சரியான செயல்பாடு

ஒப்புதல் அலகு என்பது **கானானிக்கல் செயல் பொருள்** — "மீட்டெடுப்பை ஒப்புதல்" போன்ற अस्पष्टமான குறிச்சொல்லல்ல, ஆனால் துல்லியமாக, முழுமையாக குறிப்பிடப்பட்ட செயல் ஆகும். முழு பொருளை சைன் செய்வதும் (அதிலிருந்து ஒரு டைஜஸ்ட் பெறுவதும்) நாம் பின்னர் மனிதன் *இந்த* செயலுக்கு மட்டுமே ஒப்புதல் அளித்திருப்பதை நிரூபிக்க உதவுகிறது.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## ஒரே கடிதப்பை, இரண்டு அதிகாரிகள்

ஒவ்வொரு ரசீது கூடல் கடிதப்பையாகும்: `type` புலம் கொண்ட ஒரு சாய்ந்த தகவல் மற்றும் கையொப்பம் பொருள் (`alg`, `sig`, `key_id`) அடங்கியுள்ளது, இது கையொப்பமிடப்பட்ட பைட்டுகளின் ஒரு பகுதி அல்ல. `verify_envelope` இரு ரசீது வகைகளுக்கும் பகிரப்பட்ட கட்டமைப்பு + கையொப்ப சரிபார்ப்பு ஆகும்; எந்த **நிலைபடுத்தப்பட்ட விசை பதிவேடு** இது `signature.key_id`-ஐ எதிர்பார்க்கிறது என்பது அதிகாரிகளுக்கு பிரித்துவைக்கும் காரணம்:

- **அங்கீகாரம் ரசீது** (`human.approval.v1`) — பெயரிடப்பட்ட அங்கீகரிப்பாளர், முழு canonical செயல்பாடு **மற்றும் அதன் சுருக்கம்**, `policy_version`, வெளியீடு + காலாவதி நேரடிகள். ஒருமுறை மட்டுமே பயன்படுத்துதல் சங்கிலி மட்டத்தில் கண்காணிக்கப்படுகிறது.
- **செயல் ரசீது** (`agent.action.v1`) — முகவர் அடையாளம், `run_id`, அதே canonical செயல்பாடு **சுருக்கம்**, செயல்பாட்டின் முடிவு + நேரடி, மற்றும் `parent_approval_ref`: அங்கீகாரத்தின் `receipt_hash`, பாடத்தின் சங்கிலியில் உள்ள `previous_receipt_hash` போன்ற தொடர்ச்சியான முறையில்.

பகிரப்பட்ட `action_digest` புலம் இணைப்பு பற்றியது. `key_id` கையொப்ப பொருளில் ஒரு காணொளி குறிப்பாக மட்டும் உள்ளது: அதனை வேறு நிலைபடுத்தப்பட்ட விசையின் மீது திருப்பினால் கையொப்ப சரிபார்ப்பு தோல்வியடையும், எனவே அது எந்த விளைவையும் உள்ளடக்காது.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: எங்கு பிணைப்பு உண்மையில் தீர்மானிக்கப்படுகிறது

`verify_chain` என்பது இரண்டு கையெழுத்து பரிசோதனைகளுக்கான ஒரு வசதியான அடுக்குக்கோவையல்ல. இது பகிரப்பட்ட canonical `action_digest`, அங்கீகாரத்தின் கொள்கை/சாவி/காலாவதியாக்கல் **புதியத்தன்மை**, மற்றும் அங்கீகாரத்தின் **ஒருமுறை பயன்பாடு** ஒன்றாகச் சரிபார்க்கப்படும் ஒரே இடமாகும், இப்பொழுது செயல்படுத்தப்படுகின்ற செயல் எதிர்காலத்தில்.

ஒவ்வொரு தோல்வியும் **வெவ்வேறு காரணத்துடன்** நிராகரிக்கப்படுகிறது, இதனால் நிராகரிப்பை வாசிப்பவர் அதிகாரம் காலாவதியானதா (கொள்கை மாற்றம், சாவி மாற்றம், அங்கீகாரம் காலாவதியானது, அங்கீகாரம் பயன்படுத்தப்பட்டுள்ளது) அல்லது இன்னும் செல்லுபடியான அங்கீகாரத்தின் கீழ் இயங்கும் செயல் மாற்றப்பட்டுள்ளது (தயாரக சீராக்கல்) என்பதை உணர முடியும்.


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## பைண்டிங் பிடிக்கும் விஷயங்கள்

கீழ்க்கண்ட ஒவ்வொரு நிலையும் **வெளிப்படையாக** ஒரு **தனித்த காரணத்துடன்** தோல்வி அடைகிறது. முதல் தொகுதி பாரம்பரியமாக உள்ளவை (திருட்டு, குழப்பப்பட்ட டெப்யூட்டி, மீண்டும் விளையாட்டு, எந்த அதிகாரத்திலும் வஞ்சனை, தவறான உள்ளீடு). இரண்டாமடி தொகுதி என்பது சொத்து உண்மையானதாக இருக்கும் என்று செய்யும் ஜோடி:

- **முறையீடு ஆன அதிகாரம்** — கையெழுத்து இன்னும் செல்லுபடியாக உள்ளது, ஆனால் கொள்கை பதிப்பு நகர்ந்துவிட்டது, அங்கீகாரக் விசை ரெஜிஸ்ட்ரியில் இருந்து வளைத்துவிடப்பட்டது அல்லது செயல்பாட்டு முன் அங்கீகாரம் காலாவதியானது;
- **டைகெஸ்ட் மாற்றம்** — ஒரு செல்லுபடியான கையெழுத்து செய்யப்பட்ட செயல் ரசீது, அதனுடைய `parent_approval_ref` உண்மையான அங்கீகாரம் ஒன்றை குறிக்கிறது, ஆனால் அந்த அங்கீகாரத்தின் canonical செயல் டைகெஸ்ட் செயலில் இருக்கின்ற செயலில் பொருந்தவில்லை.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## இது எதை நிரூபிக்கிறது — மற்றும் என்னை நிரூபிக்காது

**நிரூபிக்கிறது:** ஒரு பெயரிடப்பட்ட மனிதன் *இந்த துல்லியமான அரங்க செயல்பாட்டை* அங்கீகரித்துள்ளார் (முழுமையான செயல் + டைஜெஸ்ட், பின் பதிவு செய்யப்பட்ட பதிவகத்தில் இருந்து தீர்மானிக்கப்பட்ட ஒரு திறவினை கொண்டு கையொப்பமிடப்பட்டது), மற்றும் முகவர் *அந்த அங்கீகாரம் பெற்ற துல்லியமான செயல்பாட்டை* (அதே டைஜெஸ்ட், `receipt_hash` மூலம் அங்கீகாரத்துடன் கட்டுப்படுத்தப்பட்ட ரிசீப்ப்ட், பாடத்தின் சொந்த சங்க கட்டுரை விதி) ஒரே ஒரு முறையே இயக்கு விட்டார். எந்த ஒரு பக்கம் மாற்றினாலும், சங்கம் மூடப்பட்டு தவறாக முடிவடைகிறது, மறுப்புக்கான காரணம் உங்களுக்கு எது தான் உடைந்தது என்பதைச் சொல்லும்: பழைய அதிகாரம் அல்லது மாற்றிய செயல்பாடு.

**நிரூபிக்காது:** அங்கீகார UI மனிதனுக்கு அவர்கள் கையொப்பமிடப்போகும் பொருள் என்ன என்று காட்டியது (WYSIWYS என்பது தனக்கே ஒரு பிரச்சனை), திறவி மாற்றத்திற்கு முன் கட்டாயம்செய்யப்படவில்லை அல்லது திருடப்படவில்லை, அல்லது கீழடைந்த விளைவுகள் செயல் பொருந்தியது. கையொப்பம் ≠ அங்கீகாரம்: பழைய கொள்கை மீது செல்லுபடியான கையொப்பம், மாற்றப்பட்ட திறவி, காலாவதி அவகாஷம் அல்லது வேறுபட்ட டைஜெஸ்ட் இங்கு எதுவும் வழங்காது.

இரண்டு ரிசீப்ப்ட் வகைகளும் பாடத்துக்கு சொந்தமான கட்டுப்பட்டியை மற்றும் ஒரு `verify_chain` குறியீட்டு பாதையை நோக்கமாக பகிர்கின்றன: முதன்மை நோட்புக்கில் செயல்பாடு ரிசீப்ப்ட்களுக்கான பைண்டிங், மனிதனின் அங்கீகாரத்தைச் சோதிக்கும் அதே குறியீட்டு பாதை ஆகும். ஒரு சரிபார்ப்பாளர் ஒப்பந்தம், தனித்தனியான நிலைநிறுத்தப்பட்ட அதிகாரிகள், அடையாளப்படுத்தப்பட்ட செயல்பாடு டைஜெஸ்ட் மற்றும் வேறு ஒன்றும் இல்லாமல் இணைக்கப்பட்டவை.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
